In [1]:
from pathlib import Path
import sys

# This notebook lives in kuramoto/LMSSPP/notebooks.
# Add the local LMSSPP source tree without requiring package installation.
LMSSPP_SRC = Path("../src").resolve()
if not LMSSPP_SRC.exists():
    # Fallback for running the notebook from the repository root.
    LMSSPP_SRC = Path("pitch-website/public/notebooks/kuramoto/LMSSPP/src").resolve()
if str(LMSSPP_SRC) not in sys.path:
    sys.path.insert(0, str(LMSSPP_SRC))

print("LMSSPP source:", LMSSPP_SRC)


LMSSPP source: /Users/adamsobieszek/pitch/pitch-website/public/notebooks/kuramoto/LMSSPP/src


Interesting phenomena is reproduced with this settings:
```python
config = SimulationConfig(
    n_fibers=50,
    alpha=0.99,
    n_per_fiber=200,
    grid_size=512,
    domain_radius=10.0,
    make_animation=True,
    backend="numpy",
    integrator="fixed_rk2",
)
```

In [2]:
from lmsspp.dynamics.pp_cs_equilibria import SimulationConfig, make_dynamics_widget

                                # grid_size=256, animation_density_grid_size=512,
                                # dt=0.0002,
                                # animation_frame_duration_ms=90,
                                # domain_radius=15.,

config = SimulationConfig(n_fibers=10,alpha=0.99, n_per_fiber=100, 
                    color_scheme="phase_color",
                                max_steps=1000, make_animation=True)
widget = make_dynamics_widget(config)
widget                


# Non-Peszek-Poyato dynamics discovered accidentally through fixed RK2 integration with $\alpha\gg 0$, Here: $\alpha=0.99$

In [ ]:
config_interesting = SimulationConfig(
    n_fibers=10,
    alpha=0.99,
    n_per_fiber=100,
    grid_size=256,
    domain_radius=5.0,
    make_animation=True,
    integrator="fixed_rk2",
    max_steps=1000,
)
widget = make_dynamics_widget(config_interesting, 
                                second_panel="velocity",
                                )
widget                

# Predictive PP systems

In [11]:
from dataclasses import replace
from lmsspp.dynamics.pp_cs_equilibria import make_finite_horizon_gauge_averaged_widget

predictive_pp_base_config = SimulationConfig(
    n_fibers=10,
    n_per_fiber=100,
    alpha=0.99,
    K=None,
    grid_size=128,
    domain_radius=5.0,
    dt=0.01,
    dt_min=1.0e-4,
    dt_max=0.02,
    max_steps=1500,
    tol_rms=0.0,
    max_displacement_per_step=0.75,
    integrator="adaptive_rk2",
    prediction_horizon_tau=0.055,
    color_scheme="phase_color",
    make_animation=True,
    record_every=10,
)

averaged_predictive_pp_widget = make_finite_horizon_gauge_averaged_widget(
    replace(predictive_pp_base_config, 
            predictive_pp_weight=0.65,
    ),
    second_panel="velocity",
)
averaged_predictive_pp_widget

In [5]:
pure_predictive_pp_widget = make_finite_horizon_gauge_averaged_widget(
    replace(predictive_pp_base_config, predictive_pp_weight="1"),
    second_panel="velocity",
)

pure_predictive_pp_widget


# Work in progress non quadratic hamiltonian dynamic

In [6]:
from lmsspp.dynamics.pp_cs_equilibria import SimulationConfig, make_hamiltonian_exponent_widget

config=SimulationConfig(n_fibers=10,n_per_fiber=16,grid_size=128,
domain_radius=6.0,max_steps=400,make_animation=True,
hamiltonian_q=2.0,hamiltonian_epsH=0,initialization_algorithm='raw')
make_hamiltonian_exponent_widget(config)

/Users/adamsobieszek/pitch/pitch-website/public/notebooks/kuramoto/LMSSPP/src/lmsspp/dynamics/pp_cs_equilibria.py:4368: RuntimeWarning:

divide by zero encountered in log10

/opt/anaconda3/envs/manip311/lib/python3.11/site-packages/jupyter_client/session.py:721: UserWarning:

Message serialization failed with:
Out of range float values are not JSON compliant
Supporting this message is deprecated in jupyter-client 7, please make sure your message is JSON-compliant



# The Projective perturbation of the $\omega\cdot x$ natural parameter pairing

In [7]:
from lmsspp.dynamics.pp_cs_equilibria import ProjectivePeszekPoyatoDynamicsWidget, SimulationConfig

config = SimulationConfig(n_fibers=10,alpha=0.99, n_per_fiber=100, 
                                # grid_size=256, animation_density_grid_size=512,
                                # dt=0.0002,
                                # animation_frame_duration_ms=90,
                                domain_radius=5.,
                                max_steps=1000, make_animation=True)
w = ProjectivePeszekPoyatoDynamicsWidget(config)
w 

# The continuous density limit and entropic $W_{2,\nu}$ diffusion

In [8]:
# Continuous-density backend (explicit FV on grid fiber densities r_k)
from lmsspp.dynamics.pp_cs_equilibria import (
    DENSITY_DIAGNOSTIC_FIELDS,
    SimulationConfig,
    make_density_initial_condition,
    run_density_simulation,
)

density_config = SimulationConfig(
    n_fibers=6,
    alpha=0.5,
    K=1.0,
    eps_entropy=0.02,
    grid_size=64,
    domain_radius=6.0,
    dt=0.01,
    max_steps=120,
    make_animation=False,
    record_free_energy=True,
    record_entropy_balance=True,
    density_solver="explicit_fv",
    seed=2026,
)
initial = make_density_initial_condition(density_config)
result = run_density_simulation(density_config, initial)

print(f"steps={result.steps}, runtime={result.runtime_seconds:.2f}s")
if result.diagnostics.size:
    last = {name: result.diagnostics[-1, i] for i, name in enumerate(DENSITY_DIAGNOSTIC_FIELDS)}
    print(f"final rms velocity={last['rms_velocity']:.4g}")
    for key in ("total_mass", "free_energy", "entropy_H", "trace_div_A", "fisher_information"):
        print(f"{key}={last[key]:.4g}")
result

/Users/adamsobieszek/pitch/pitch-website/public/notebooks/kuramoto/LMSSPP/src/lmsspp/dynamics/pp_cs_equilibria.py:2362: RuntimeWarning:

density_solver='explicit_fv' with eps_entropy > 0 can develop spurious nonlocal mass; prefer 'chang_cooper' or 'split_implicit_diffusion'.



steps=120, runtime=0.81s
final rms velocity=1.055
total_mass=1
free_energy=1.081
entropy_H=0.04939
trace_div_A=3.205
fisher_information=43.66


DensitySimulationResult(initial=DensityInitialCondition(r_fiber=array([[[2.58794265e-141, 2.49571220e-138, 1.88477984e-135, ...,
         1.42450218e-153, 4.58464330e-157, 1.15551146e-160],
        [7.99277347e-137, 7.70792283e-134, 5.82107887e-131, ...,
         4.39952688e-149, 1.41595160e-152, 3.56875812e-156],
        [1.93315480e-132, 1.86426002e-129, 1.40790260e-126, ...,
         1.06408202e-144, 3.42466060e-148, 8.63149935e-152],
        ...,
        [9.75767356e-062, 9.40992449e-059, 7.10644279e-056, ...,
         5.37099510e-074, 1.72861067e-077, 4.35678265e-081],
        [1.00575389e-063, 9.69910307e-061, 7.32483254e-058, ...,
         5.53605240e-076, 1.78173301e-079, 4.49067195e-083],
        [8.11826883e-066, 7.82894573e-063, 5.91247624e-060, ...,
         4.46860431e-078, 1.43818360e-081, 3.62479156e-085]],

       [[3.01925120e-135, 1.84768260e-130, 8.70651320e-126, ...,
         5.55608249e-051, 4.04866591e-053, 2.27166334e-055],
        [1.22190900e-132, 7.47768184e-1

In [9]:
# Continuous-density widget (heatmap playback; click Precompute, then Play/Step/slider)
from lmsspp.dynamics.pp_cs_equilibria import SimulationConfig, make_continuous_density_widget

density_widget_config = SimulationConfig(
    n_fibers=8,
    alpha=0.8,
    K=1.0,
    eps_entropy=0.03,
    grid_size=128,
    
    domain_radius=8.0,
    dt=0.008,
    max_steps=300,

    
    make_animation=True,
    trajectory_frame_count=0,
    record_free_energy=True,
    record_entropy_balance=True,
    density_solver="split_implicit_diffusion",
    seed=2026,
)
density_widget = make_continuous_density_widget(density_widget_config)
density_widget